# AI for Climate Risk: calibrating AIFS rainfall forecasts with isotonic distributional regression

**Workshop notebook · Annual Conference of the South African Statistical Association (SASA) 2026**
*AI for Climate Risk: Statistical Foundations for Forecasting in Data-Scarce Environments* · Prof Rendani Mbuvha, University of the Witwatersrand

ECMWF's **AIFS-Single** is a machine-learning weather model. It gives one rainfall number per grid box and day. In this notebook we turn that number into a **calibrated predictive distribution** with **isotonic distributional regression (IDR)**, score it against **GPM IMERG** satellite rainfall with the **CRPS**, and compare it with a 21-year **climatology**. We then ask how much the answer can be trusted with so little data, and what the forecasts are worth to people who must decide whether to act.

The question throughout: **does AIFS add trustworthy information about tomorrow's rain beyond climatology, once its errors are corrected?**

| Section | What you do | Workshop objective |
|---|---|---|
| 1–3 | Load climate zones, IMERG and AIFS; see how little data one grid box has | Limited and noisy data |
| 4 | Fit IDR; step through PAVA by hand | Statistical foundations |
| 5–6 | Calibrate every zone and lead day; score with the CRPS; decompose it into MCB − DSC + UNC | Statistical foundations |
| 7 | Quantify uncertainty: learning curves, block bootstrap, a shuffle control | Uncertainty quantification |
| 8–9 | Maps and climate-zone tables of where AIFS adds value | Case studies across African NMHSs |
| 10 | Heavy-rain probabilities: reliability, ROC and the cost–loss value score | Risk indices and early warning |
| 11 | Discussion questions | Open challenges |

The companion note and the interactive app follow the same sections. Run the cells from top to bottom; the whole notebook takes about 10–15 minutes for all of Africa, and less for a smaller region (see *Configuration*).

### The data

| | Source | Grid | Period |
|---|---|---|---|
| **Forecast (training)** | AIFS-Single, 00 UTC runs, 24 h rainfall (`aifs_tp24h_africa.zarr`) | 0.25° | runs Mar 2024 – Jan 2025 |
| **Forecast (testing)** | the same (`aifs_tp24h_africa_2026.zarr`) | 0.25° | runs Jan – Oct 2026 |
| **Observations (training)** | GPM IMERG V07B **Final**, daily 00–24 UTC (`imerg_daily_africa_2024_2025.zarr`) | 0.1° | Mar 2024 – Feb 2025 |
| **Observations (testing)** | GPM IMERG V07, **Late** run where Final is not yet available (`imerg_daily_africa_2026.zarr`) | 0.1° | Jan – Oct 2026 |
| **Climate zones** | Köppen–Geiger (Beck et al. 2023) | 0.1° | – |
| **Climatology** | CRPS of an IMERG climatological forecast: all days within ±15 days of the date in 2000–2020, about 650 values (`imerg_clim_crps_daily_africa_2026.zarr`) | 0.1° | the 2026 test days |

**Lead day.** `step = 48 h` is the rain from 24 h to 48 h after the 00 UTC start time, called lead day 2. A run started on day *d* is therefore matched with IMERG for day *d* + 1. We use lead days 2–5.

**Units.** All CRPS calculations use mm/hr (the daily mean rate). Multiply by 24 for mm/day.

**Two features of these data matter for everything that follows.** First, the record is short: under a year of AIFS forecasts to learn from, against 21 years of observations for the climatology. Second, the "truth" is itself uncertain and changes between periods: training uses gauge-adjusted IMERG Final, while every 2026 test day uses the Late run, which is not gauge-adjusted. Some of the test error may come from that change rather than from AIFS.

### About the datasets

**AIFS-Single (the forecast).** The deterministic version of ECMWF's Artificial Intelligence Forecasting System, a machine-learning weather model that runs alongside ECMWF's physics-based IFS. A graph neural network encodes the atmosphere, a transformer processes it, and a decoder maps it back to a global grid. It steps forward 6 hours at a time. It was trained on decades of ERA5 reanalysis and fine-tuned on operational analyses, and each forecast starts from ECMWF's analysis of the current atmosphere. Rainfall is one of its outputs, on a grid of roughly 0.25° (about 28 km). "Single" means one run with no ensemble, so no built-in uncertainty, which is what calibration supplies. AIFS became an operational ECMWF product in 2025 and is updated from time to time; the 2024 training forecasts predate that.

**GPM IMERG (the "truth").** NASA's Integrated Multi-satellitE Retrievals for the Global Precipitation Measurement mission, version 7: every 30 minutes at 0.1°, summed here to daily 00–24 UTC totals. Microwave sensors on a constellation of satellites estimate rain as they pass over, all calibrated to the GPM Core Observatory's radar and imager. Infrared images from geostationary satellites fill the gaps between passes. It is issued in three runs: **Early** (about 4 hours after real time), **Late** (about half a day after) and **Final** (about 3.5 months later, adjusted to monthly rain-gauge totals). Satellite estimates can misjudge drizzle, warm-cloud and mountain rain, and the most intense downpours, and gauge adjustment helps only where gauges exist.

**IMERG climatology (the reference).** For each grid box and day, all IMERG daily totals within ±15 days of the date in 2000–2020: 31 × 21, about 650 values, used as an ensemble forecast. It knows the place and the season but nothing about today's weather. A forecast that cannot beat it adds no information.

**Köppen–Geiger zones (the pooling groups).** A climate classification from long-term monthly temperature and rainfall. The first letter is the main group (A tropical, B arid, C temperate, D continental, E polar). The second is the rainfall pattern (f no dry season, m monsoon, w dry winter, s dry summer; W desert and S steppe for arid climates). A third letter, where present, gives temperature (h hot, k cold). So **Aw** is tropical savannah with a dry winter, and **BWh** is hot desert. The map (Beck et al. 2023) is on IMERG's 0.1° grid.

## 0 · Setup

In [ ]:
# Colab already has numpy, xarray, pandas, matplotlib, scipy and numba. Add zarr v3 (the format
# of the files), gcsfs (to read Google Cloud Storage) and cartopy (for maps).
!pip -q install "zarr>=3" gcsfs cartopy

In [ ]:
import tempfile
import time
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gcsfs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import BoundaryNorm, ListedColormap
from numba import njit, prange
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)   # all-NaN means over the sea
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "axes.edgecolor": "#898781"})

### Configuration

These are the only settings you need to change. Smaller `REGION` boxes run faster:

| Region | `lat` | `lon` |
|---|---|---|
| All of Africa (default) | (-35, 38) | (-18, 52) |
| East Africa | (-12, 15) | (28, 52) |
| West Africa | (4, 20) | (-18, 16) |
| Southern Africa | (-35, -8) | (10, 41) |

In [ ]:
BUCKET = "acmad-tutorial"      # public Google Cloud Storage bucket with all the data
FILES = dict(
    aifs_train="aifs_tp24h_africa.zarr",
    aifs_test="aifs_tp24h_africa_2026.zarr",
    imerg_train="imerg_daily_africa_2024_2025.zarr",
    imerg_test="imerg_daily_africa_2026.zarr",
    koppen="koppen_geiger_africa_0p1.nc",
    clim_test="imerg_clim_crps_daily_africa_2026.zarr",   # CRPS of IMERG climatology, 2026 days
)

REGION = dict(lat=(-35.0, 38.0), lon=(-18.0, 52.0))   # degrees N, degrees E
LEAD_DAYS = [2, 3, 4, 5]
EVENT_MM_DAY = 10.0        # threshold for the "heavy rain" probability, in mm/day

# Units: the calculations use mm/hr (daily-mean rate). Amounts in mm/day are divided by 24.
MM_DAY = 1 / 24                  # 1 mm/day expressed in mm/hr
EVENT = EVENT_MM_DAY * MM_DAY    # the threshold in mm/hr

# Calibration settings, as in the AfriClimate AI weather post-processing code.
RES = 0.1 * MM_DAY         # rainfall is rounded to 0.1 mm/day (in mm/hr) before fitting
ZONE_SAMPLE_LOCS = 1_000   # grid boxes sampled at random from each climate zone for training
NUM_DATA_PTS = 50_000      # most (forecast, observation) pairs used to train each zone's model
ZONE_MIN_VALID = 100       # zones with fewer training pairs than this are skipped
SEED = 0                   # random seed, so results can be repeated

# Workshop additions
PROB_BINS = np.linspace(0, 1, 21)   # 0.05-wide probability bins for reliability, ROC and value
DEMO_SAMPLE_BOXES = 3_000           # grid boxes of the demo zone kept for the Section 7 experiments

### Helper functions

Functions to open files in the bucket, the Köppen–Geiger legend, and a map-plotting helper.

In [ ]:
def open_zarr(name):
    # Open a zarr store in the public bucket, without logging in.
    return xr.open_zarr(f"gs://{BUCKET}/{name}", decode_timedelta=False, storage_options={"token": "anon"})


# Factor to mm/hr for each `units` attribute; "mm" is a 24 h total (AIFS tp_24h).
TO_MM_HR = {"mm/hr": 1.0, "mm/h": 1.0, "mm/day": 1 / 24, "mm": 1 / 24}


def in_mm_hr(da):
    units = da.attrs.get("units")
    if units not in TO_MM_HR:
        raise ValueError(f"{da.name}: unknown units {units!r}")
    return da * TO_MM_HR[units]


def load_koppen():
    local = Path(tempfile.gettempdir()) / FILES["koppen"]
    if not local.exists():
        gcsfs.GCSFileSystem(token="anon").get(f"{BUCKET}/{FILES['koppen']}", str(local))
    return xr.open_dataset(local)["kg_class"].load()


# Köppen–Geiger classes (Beck et al. 2023): id -> (code, description, standard colour)
KG_LEGEND = {
    1: ("Af", "Tropical rainforest", "#0000ff"),       2: ("Am", "Tropical monsoon", "#0078ff"),
    3: ("Aw", "Tropical savannah", "#46aafa"),         4: ("BWh", "Hot desert", "#ff0000"),
    5: ("BWk", "Cold desert", "#ff9696"),              6: ("BSh", "Hot steppe", "#f5a500"),
    7: ("BSk", "Cold steppe", "#ffdc64"),              8: ("Csa", "Mediterranean, hot summer", "#ffff00"),
    9: ("Csb", "Mediterranean, warm summer", "#c8c800"), 10: ("Csc", "Mediterranean, cold summer", "#969600"),
    11: ("Cwa", "Dry winter, hot summer", "#96ff96"),  12: ("Cwb", "Highland, dry winter", "#64c864"),
    13: ("Cwc", "Dry winter, cold summer", "#329632"), 14: ("Cfa", "Humid subtropical", "#c8ff50"),
    15: ("Cfb", "Oceanic", "#64ff50"),                 16: ("Cfc", "Subpolar oceanic", "#32c800"),
    17: ("Dsa", "Continental, dry hot summer", "#ff00ff"), 18: ("Dsb", "Continental, dry warm summer", "#c800c8"),
    19: ("Dsc", "Continental, dry cool summer", "#963296"), 20: ("Dsd", "Continental, dry summer, very cold", "#966496"),
    21: ("Dwa", "Continental, dry winter, hot", "#aaafff"), 22: ("Dwb", "Continental, dry winter, warm", "#5a78dc"),
    23: ("Dwc", "Continental, dry winter, cool", "#4b50b4"), 24: ("Dwd", "Continental, dry winter, very cold", "#320087"),
    25: ("Dfa", "Continental, humid, hot", "#00ffff"), 26: ("Dfb", "Continental, humid, warm", "#37c8ff"),
    27: ("Dfc", "Subarctic", "#007d7d"),               28: ("Dfd", "Subarctic, very cold", "#00465f"),
    29: ("ET", "Tundra", "#b2b2b2"),                   30: ("EF", "Ice cap", "#666666"),
}

ORANGE, BLUE, GREY = "#eb6834", "#2a78d6", "#898781"   # raw AIFS, calibrated (IDR), climatology


def map_panels(fields, titles, cmap="Blues", vmin=None, vmax=None, norm=None, label="",
               ncols=4, suptitle=None, extend="max"):
    # Plot a list of 2-D fields (on the region's 0.1° grid) side by side, sharing one colour bar.
    n = len(fields)
    nrows = int(np.ceil(n / ncols))
    ncols = min(n, ncols)
    width = 3.6 * ncols * (np.ptp(REGION["lon"]) / np.ptp(REGION["lat"])) ** 0.5
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, 3.6 * nrows + 0.6),
                             subplot_kw={"projection": ccrs.PlateCarree()}, squeeze=False)
    for ax, f, t in zip(axes.flat, fields, titles):
        im = ax.pcolormesh(GRID_LON, GRID_LAT, f, cmap=cmap, vmin=None if norm else vmin,
                           vmax=None if norm else vmax, norm=norm, shading="auto",
                           transform=ccrs.PlateCarree())
        ax.coastlines("50m", lw=0.5)
        ax.add_feature(cfeature.BORDERS, lw=0.3, edgecolor="#52514e")
        ax.set_extent([*REGION["lon"], *REGION["lat"]], crs=ccrs.PlateCarree())
        ax.set_title(t, fontsize=10)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    fig.colorbar(im, ax=axes, shrink=0.8, label=label, extend=extend)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    plt.show()

## 1 · Climate zones: pooling grid boxes ("space for time")

One grid box has under a year of training days, with very few heavy-rain days among them: far too few to estimate a probability of heavy rain. We therefore **pool** grid boxes that share a broadly similar rain climate. Every grid box in the same Köppen–Geiger zone (for example *Aw, tropical savannah*) gets the same model, trained on pairs drawn from up to 1,000 grid boxes across the zone. This gives each model tens of thousands of training pairs, including many heavy-rain days.

The price is an assumption: that grid boxes in a zone share one forecast-error behaviour. Section 7 tests how much the choice matters.

In [ ]:
kg = load_koppen().sel(lat=slice(*REGION["lat"]), lon=slice(*REGION["lon"]))
GRID_LAT, GRID_LON = kg.lat.values, kg.lon.values
KG = kg.values
LAND = KG > 0                       # class 0 = sea; we calibrate land only
PIX_ZONE = KG[LAND]                 # climate zone of every land grid box
NPIX = int(LAND.sum())


def to_map(values):
    # Put a per-grid-box vector (length NPIX) back on the 2-D map.
    out = np.full(LAND.shape, np.nan, dtype=np.float32)
    out[LAND] = values
    return out


ZONES = [int(z) for z in np.unique(PIX_ZONE)]
zone_table = pd.DataFrame(
    [(KG_LEGEND[z][0], KG_LEGEND[z][1], int((PIX_ZONE == z).sum())) for z in ZONES],
    index=pd.Index(ZONES, name="id"), columns=["code", "climate", "grid boxes"])
DEMO_ZONE = 3 if 3 in ZONES else int(zone_table["grid boxes"].idxmax())   # Aw, or the largest zone
print(f"{NPIX:,} land grid boxes at 0.1° in {len(ZONES)} climate zones; "
      f"demo zone for Sections 4 and 7: {KG_LEGEND[DEMO_ZONE][0]} ({KG_LEGEND[DEMO_ZONE][1]})")
zone_table

In [ ]:
present = sorted(ZONES)
cmap_kg = ListedColormap([KG_LEGEND[z][2] for z in present])
norm_kg = BoundaryNorm(np.arange(len(present) + 1) - 0.5, len(present))
zone_idx = np.full(LAND.shape, np.nan)
zone_idx[LAND] = np.searchsorted(present, PIX_ZONE)

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={"projection": ccrs.PlateCarree()})
ax.pcolormesh(GRID_LON, GRID_LAT, zone_idx, cmap=cmap_kg, norm=norm_kg, shading="auto")
ax.coastlines("50m", lw=0.5)
ax.add_feature(cfeature.BORDERS, lw=0.3, edgecolor="#52514e")
ax.set_extent([*REGION["lon"], *REGION["lat"]], crs=ccrs.PlateCarree())
handles = [plt.Rectangle((0, 0), 1, 1, color=KG_LEGEND[z][2]) for z in present]
ax.legend(handles, [f"{KG_LEGEND[z][0]}  {KG_LEGEND[z][1]}" for z in present],
          loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
ax.set_title("Köppen–Geiger climate zones: one calibration model per zone")
plt.show()

## 2 · Observations (IMERG) and the reference forecast

Models are **trained** on forecasts from March 2024 to January 2025 and **tested** on the 2026 forecasts. The test period is never seen during training, so the test scores show how calibration would perform in real use.

The **reference forecast** is the IMERG climatology: for each day and grid box, the set of all IMERG values within ±15 days of that date in 2000–2020 (31 days × 21 years, about 650 values). Its CRPS has been calculated in advance for the 2026 test days. Any forecast worth issuing must beat it.

In [ ]:
def to_pixels(da, regrid=False, block=32):
    # (time, lat, lon) field -> (time, land grid boxes) float32, a block of days at a time.
    # With regrid=True the field is first interpolated (bilinear) onto the 0.1° IMERG grid.
    out = np.empty((da.sizes["time"], NPIX), dtype=np.float32)
    for s in range(0, da.sizes["time"], block):
        blk = da.isel(time=slice(s, s + block))
        if regrid:
            blk = blk.interp(lat=GRID_LAT, lon=GRID_LON, method="linear")
        out[s:s + block] = blk.values[:, LAND]
    return out


def load_imerg(name):
    ds = open_zarr(name)
    var = "precip" if "precip" in ds else "precipitation"
    da = in_mm_hr(ds[var]).sel(lat=slice(*REGION["lat"]), lon=slice(*REGION["lon"]))
    # same 0.1° boxes as the climate-zone map (grid boxes are matched by position)
    assert da.shape[1:] == LAND.shape and np.allclose(da.lat, GRID_LAT, atol=1e-3) \
        and np.allclose(da.lon, GRID_LON, atol=1e-3), f"{name} is not on the IMERG 0.1° grid"
    return to_pixels(da), pd.DatetimeIndex(da.time.values)


t0 = time.time()
OBS_TRAIN, OBS_TRAIN_DATES = load_imerg(FILES["imerg_train"])
OBS_TEST, OBS_TEST_DATES = load_imerg(FILES["imerg_test"])
print(f"IMERG train: {OBS_TRAIN_DATES[0]:%d %b %Y} – {OBS_TRAIN_DATES[-1]:%d %b %Y}"
      f"   test: {OBS_TEST_DATES[0]:%d %b %Y} – {OBS_TEST_DATES[-1]:%d %b %Y}"
      f"   ({time.time() - t0:.0f} s)")
gaps = pd.date_range(OBS_TEST_DATES[0], OBS_TEST_DATES[-1]).difference(OBS_TEST_DATES)
test_ds = open_zarr(FILES["imerg_test"])
if "imerg_run" in test_ds:
    run = test_ds["imerg_run"].values
    print(f"2026 IMERG: {(run == 1).sum()} days Final run, {(run == 2).sum()} days Late run "
          "(training used the Final run throughout)")
if len(gaps):
    print("Days with no IMERG (skipped):", ", ".join(f"{d:%d %b}" for d in gaps))


def load_climatology(name, dates):
    # CRPS of the IMERG climatological forecast in mm/hr (and its probability of more
    # than EVENT_MM_DAY), one row per date in `dates`. None if the file is not available.
    ds = open_zarr(name).sel(lat=slice(*REGION["lat"]), lon=slice(*REGION["lon"]))
    ds = ds.reindex(time=dates)                      # days not in the file -> NaN
    p_name = f"p_clim_{EVENT_MM_DAY:g}mm_day"
    return dict(crps=to_pixels(in_mm_hr(ds["crps_clim"])),
                p_event=to_pixels(ds[p_name]) if p_name in ds else None)


CLIM_TEST = load_climatology(FILES["clim_test"], OBS_TEST_DATES)
print("Reference forecast:", "IMERG climatology (2000–2020, ±15 days)" if CLIM_TEST is not None
      else "no climatology file for these days, using each zone's training-period climatology")

## 3 · AIFS forecasts: regridding to the IMERG grid

AIFS is on a 0.25° grid and IMERG on a 0.1° grid. We **interpolate AIFS onto the IMERG grid** (bilinear), so the forecast and the observation refer to the same 0.1° box. This is regridding, not downscaling: interpolation adds no fine-scale spatial detail. The calibration in Section 4 then corrects the forecast's distribution towards what IMERG records in 0.1° boxes. We load only lead days 2–5.

In [ ]:
def load_aifs(name):
    raw = open_zarr(name)["tp_24h"]
    ds = raw.rename(latitude="lat", longitude="lon")
    ds = ds.sel(step=[24.0 * d for d in LEAD_DAYS],
                lat=slice(REGION["lat"][0] - 0.5, REGION["lat"][1] + 0.5),
                lon=slice(REGION["lon"][0] - 0.5, REGION["lon"][1] + 0.5))
    da = ds.clip(min=0).load()            # mm per 24 h; remove tiny negative values
    return da * TO_MM_HR[raw.attrs["units"]]   # 24 h total (mm) -> mm/hr


t0 = time.time()
AIFS_TRAIN = load_aifs(FILES["aifs_train"])
AIFS_TEST = load_aifs(FILES["aifs_test"])
print(f"AIFS train runs: {AIFS_TRAIN.time.size} ({str(AIFS_TRAIN.time.values[0])[:10]} – "
      f"{str(AIFS_TRAIN.time.values[-1])[:10]})   test runs: {AIFS_TEST.time.size} "
      f"({str(AIFS_TEST.time.values[0])[:10]} – {str(AIFS_TEST.time.values[-1])[:10]})"
      f"   ({time.time() - t0:.0f} s)")


def paired(aifs, obs, obs_dates, lead):
    # Forecast and observation tables (days × grid boxes) for one lead day, matched on the
    # date the forecast is valid for: start date + (lead - 1) days.
    fc = aifs.sel(step=24.0 * lead)
    valid = pd.DatetimeIndex(fc.time.values) + pd.Timedelta(days=lead - 1)
    pos = obs_dates.get_indexer(valid)
    keep = pos >= 0
    X = to_pixels(fc.isel(time=np.flatnonzero(keep)), regrid=True)
    return X, obs[pos[keep]], valid[keep], pos[keep]

In [ ]:
# Check: one forecast before and after regridding, next to what IMERG observed.
lead, k = LEAD_DAYS[0], min(100, AIFS_TRAIN.time.size - 1)
X, Y, vd, _ = paired(AIFS_TRAIN.isel(time=slice(k, k + 1)), OBS_TRAIN, OBS_TRAIN_DATES, lead)
rain_levels = [0, 0.5, 1, 2, 5, 10, 20, 35, 50, 75, 100]
rain_norm = BoundaryNorm(rain_levels, 256, extend="max")
coarse = AIFS_TRAIN.sel(step=24.0 * lead).isel(time=k).interp(
    lat=GRID_LAT, lon=GRID_LON, method="nearest").values
map_panels([np.where(LAND, coarse, np.nan) / MM_DAY, to_map(X[0] / MM_DAY), to_map(Y[0] / MM_DAY)],
           ["AIFS on its 0.25° grid", "AIFS regridded to 0.1°", "IMERG observed"],
           cmap="YlGnBu", norm=rain_norm, label="mm/day", ncols=3,
           suptitle=f"Rain on {vd[0]:%d %b %Y} (lead day {lead})")

## 4 · The calibration method: isotonic distributional regression (IDR)

Having regridded AIFS to the IMERG grid, we calibrate it: turn each raw forecast into a **predictive distribution** for the rain IMERG will record.

**The statistical idea.** For a forecast value $x$ and observed rain $Y$, distributional regression targets the whole conditional CDF $F(y \mid x) = \mathbb{P}(Y \le y \mid X = x)$, not just the mean. IDR (Henzi, Ziegel & Gneiting, 2021) makes one assumption: a larger forecast means a stochastically larger outcome,

$$x \le x' \;\Longrightarrow\; F(z \mid x) \ge F(z \mid x') \quad \text{for every threshold } z .$$

Among all collections of CDFs that respect this order, IDR picks the one with the smallest mean CRPS on the training pairs. Because the CRPS is an integral of Brier scores over thresholds, the problem **separates by threshold**. At each $z$ the fitted values solve a decreasing isotonic least-squares regression of the indicators $\mathbf 1\{y_i \le z\}$ on $x_i$, which the **pool-adjacent-violators algorithm (PAVA)** solves exactly.

**Properties.** There is no distribution family and no tuning parameter; the result can never give probability to negative rain. The predictive distributions are monotone **step functions** (not smooth curves) supported on the observed training values. IDR **does not extrapolate**: beyond the largest training forecast the distribution stays fixed.

### PAVA by hand

Five forecasts sorted from driest to wettest, with indicators $b_i = \mathbf 1\{y_i \le 20\text{ mm}\}$. The fitted probabilities must not increase from left to right. Whenever a neighbouring pair breaks the order, pool them into their weighted mean.

In [ ]:
def pava_decreasing_verbose(b, w=None):
    # Pool-adjacent-violators for a non-increasing fit, printing every merge.
    w = np.ones(len(b)) if w is None else np.asarray(w, float)
    blocks = []                                   # each block: [value, weight, count]
    for v, wt in zip(b, w):
        blocks.append([float(v), float(wt), 1])
        while len(blocks) > 1 and blocks[-2][0] < blocks[-1][0]:
            (v1, w1, c1), (v2, w2, c2) = blocks[-2], blocks.pop()
            blocks[-1] = [(v1 * w1 + v2 * w2) / (w1 + w2), w1 + w2, c1 + c2]
            print("  merge ->", " | ".join(f"{bv:.2f}×{bc}" for bv, _, bc in blocks))
    return np.repeat([bv for bv, _, _ in blocks], [bc for _, _, bc in blocks])


x_toy = np.array([5, 10, 15, 20, 25])            # AIFS forecasts, mm/day (sorted)
b_toy = np.array([1, 0, 1, 1, 0])                # 1 = observed rain stayed at or below 20 mm
print("indicators:", b_toy)
fit = pava_decreasing_verbose(b_toy)
print("fitted P(Y <= 20 mm | x):", np.round(fit, 3), "  exceedance:", np.round(1 - fit, 3))

The engine below does the same thing for every threshold at once, on integer levels of 0.1 mm/day. You don't need to read it to follow the tutorial.

In [ ]:
@njit(cache=True)
def _antitonic(y, w):
    # Pool-adjacent-violators: weighted least-squares fit of y that never increases.
    n = y.size
    val, wt, cnt = np.empty(n), np.empty(n), np.empty(n, np.int64)
    k = -1
    for i in range(n):
        k += 1
        val[k], wt[k], cnt[k] = y[i], w[i], 1
        while k > 0 and val[k - 1] < val[k]:
            tw = wt[k - 1] + wt[k]
            val[k - 1] = (val[k - 1] * wt[k - 1] + val[k] * wt[k]) / tw
            wt[k - 1], cnt[k - 1] = tw, cnt[k - 1] + cnt[k]
            k -= 1
    out, i = np.empty(n), 0
    for b in range(k + 1):
        for _ in range(cnt[b]):
            out[i] = val[b]
            i += 1
    return out


@njit(parallel=True, cache=True)
def _antitonic_columns(F, w):
    out = np.empty_like(F)
    for t in prange(F.shape[1]):
        out[:, t] = _antitonic(F[:, t].copy(), w)
    return out


assert np.allclose(_antitonic(b_toy.astype(float), np.ones(5)), fit)   # same answer as the hand version


def fit_idr(x, y):
    # Fit IDR to forecast x and observed y (mm/hr). Everything is stored in whole
    # multiples of RES (0.1 mm/day), so a forecast of 12.3 mm/day is level 123.
    xi, yi = np.rint(x / RES).astype(np.int64), np.rint(y / RES).astype(np.int64)
    ux, gx, nx = np.unique(xi, return_inverse=True, return_counts=True)   # forecast levels
    z, gy = np.unique(yi, return_inverse=True)                            # observed levels
    counts = np.bincount(gx * z.size + gy, minlength=ux.size * z.size).reshape(ux.size, z.size)
    F = np.cumsum(counts, axis=1) / nx[:, None]       # P(obs <= z | forecast = x), from the data
    F = _antitonic_columns(F, nx.astype(np.float64))  # wetter forecast => never a drier outlook
    clim = np.cumsum(counts.sum(axis=0)) / xi.size    # zone climatology (ignores the forecast)
    return dict(x=ux, z=z, cdf=F, clim=clim)


def predict_cdf(model, xi):
    # Predictive CDF (rows) at the observed levels model["z"] for forecast levels xi:
    # interpolated linearly between the nearest training forecasts, constant beyond them.
    pos = np.interp(xi, model["x"], np.arange(model["x"].size))
    il = np.floor(pos).astype(np.int64)
    ir = np.minimum(il + 1, model["x"].size - 1)
    w = (pos - il)[:, None]
    return (1 - w) * model["cdf"][il] + w * model["cdf"][ir]


def _crps_from_cdf(F):
    # F: (..., G) CDF on the levels 0..G-1 (constant between levels, 1 after the last).
    # Returns CRPS (mm/hr) for an observation at each level: sum F^2 below it and
    # (1-F)^2 from it upwards (exact for these step-function CDFs).
    sq = F * F
    below = np.cumsum(sq, axis=-1) - sq
    above = np.cumsum(((1 - F) ** 2)[..., ::-1], axis=-1)[..., ::-1]
    return (RES * (below + above)).astype(np.float32)


EV = int(round(EVENT / RES))   # the heavy-rain threshold as a level


def lookup_tables(model, max_obs_level):
    # Pre-compute everything for one zone's model, so verifying millions of
    # forecasts is a table lookup: crps[forecast level, observed level], etc.
    R = int(model["x"][-1]) + 1                       # forecast levels 0..largest training forecast
    G = max(int(max_obs_level), int(model["z"][-1])) + 1
    rows = predict_cdf(model, np.arange(R))           # (R, number of observed levels)
    j = np.searchsorted(model["z"], np.arange(G), side="right") - 1
    on_grid = lambda F: np.where(j >= 0, F[..., np.maximum(j, 0)], 0.0)
    Fg = on_grid(rows).astype(np.float32)             # (R, G)
    Fc = on_grid(model["clim"])                       # (G,)
    first_reaching = lambda p: model["z"][np.argmax(rows >= p, axis=1)] * RES
    return dict(R=R,
                crps=_crps_from_cdf(Fg), crps_clim=_crps_from_cdf(Fc),
                p_event=1 - Fg[:, min(EV, G - 1)] if EV < G else np.zeros(R, np.float32),
                p_event_clim=float(1 - Fc[min(EV, G - 1)]) if EV < G else 0.0,
                q10=first_reaching(0.1), q50=first_reaching(0.5), q90=first_reaching(0.9))

### What IDR learns, and how little one grid box knows

One model: the demo zone (tropical savannah, Aw, or the largest zone if your region has none), lead day 2. Each line on the left is the **predicted rain distribution** for one raw AIFS forecast amount. As the forecast gets wetter the curves move right, as the isotonic rule requires. Even a forecast of **0 mm** keeps a real chance of rain. On the right are the calibrated median and 10–90% range against the raw forecast.

The printout below the plot compares the data budget of one grid box with the pooled zone.

In [ ]:
rng = np.random.default_rng(SEED)
X_tr, Y_tr, _, _ = paired(AIFS_TRAIN, OBS_TRAIN, OBS_TRAIN_DATES, LEAD_DAYS[0])
zp = np.flatnonzero(PIX_ZONE == DEMO_ZONE)
locs = rng.choice(zp, size=min(zp.size, ZONE_SAMPLE_LOCS), replace=False)
x, y = X_tr[:, locs].ravel(), Y_tr[:, locs].ravel()
ok = np.flatnonzero(np.isfinite(x) & np.isfinite(y))
ok = rng.choice(ok, size=min(ok.size, NUM_DATA_PTS), replace=False)
demo = fit_idr(x[ok], y[ok])
tab = lookup_tables(demo, demo["z"][-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3))
amounts = [0, 2, 5, 10, 20, 40]
shades = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#0d366b"]
for a, c in zip(amounts, shades):
    F = predict_cdf(demo, np.array([round(a * MM_DAY / RES)]))[0]
    ax1.step(demo["z"] * RES / MM_DAY, F, where="post", color=c, lw=2, label=f"AIFS = {a} mm/day")
ax1.set(xscale="symlog", xlim=(0, 150), ylim=(0, 1.02), xlabel="observed rain (mm/day)",
        ylabel="probability rain ≤ amount", title="Predicted distribution for each forecast amount")
ax1.legend(frameon=False, fontsize=9, loc="lower right")

xs = np.arange(0, min(tab["R"], int(60 * MM_DAY / RES)))
day = lambda v: v / MM_DAY                              # mm/hr -> mm/day, for display
ax2.fill_between(day(xs * RES), day(tab["q10"][xs]), day(tab["q90"][xs]), color=BLUE, alpha=0.18, lw=0,
                 label="10–90% range")
ax2.plot(day(xs * RES), day(tab["q50"][xs]), color=BLUE, lw=2, label="median")
ax2.plot([0, 60], [0, 60], color=GREY, lw=1, ls="--", label="raw forecast at face value (1:1)")
ax2.set(xlim=(0, 60), ylim=(0, 60), xlabel="raw AIFS forecast (mm/day)", ylabel="observed rain (mm/day)",
        title="Calibrated median and range vs raw forecast")
ax2.legend(frameon=False, fontsize=9, loc="upper left")
fig.suptitle(f"IDR model for zone {KG_LEGEND[DEMO_ZONE][0]} ({KG_LEGEND[DEMO_ZONE][1]}), "
             f"lead day {LEAD_DAYS[0]}, {ok.size:,} training pairs", fontsize=11)
plt.tight_layout()
plt.show()

n_days = X_tr.shape[0]
heavy_share = float((y[ok] >= 50 * MM_DAY).mean())
print(f"One grid box alone: {n_days} training days, of which about {heavy_share * n_days:.1f} "
      f"had at least 50 mm.\nPooled zone sample: {ok.size:,} pairs, of which "
      f"{int((y[ok] >= 50 * MM_DAY).sum()):,} had at least 50 mm.")
del X_tr, Y_tr

## 5 · Calibrate and verify every lead day

For each lead day:

1. Match training forecasts with IMERG and fit **one IDR model per climate zone**.
2. Apply each model to **every** grid box in its zone, on every test day.
3. Score raw AIFS, calibrated AIFS and the **IMERG climatology** from Section 2. For a single-number forecast the CRPS is the absolute error, so raw AIFS is scored by $|x - y|$.

We also keep the counts needed for the heavy-rain reliability, ROC and value analyses in Section 10, the daily sums needed for the bootstrap in Section 7, and a sample of the demo zone's test pairs for the experiments in Section 7. Heavy-rain events and raw yes/no forecasts are defined on the same 0.1 mm/day levels as the model, so all three forecasts are judged on exactly the same event.

In [ ]:
def fit_zone_models(X, Y, rng):
    # One IDR model per climate zone, trained on pairs pooled from grid boxes in that zone.
    # Also returns the pairs each model was fitted on (forecast, observation, day index).
    models, samples = {}, {}
    for z in ZONES:
        zp = np.flatnonzero(PIX_ZONE == z)
        # pick whole grid boxes (each box with all its days), then thin the pairs
        locs = rng.choice(zp, size=min(zp.size, ZONE_SAMPLE_LOCS), replace=False)
        x, y = X[:, locs].ravel(), Y[:, locs].ravel()
        ok = np.flatnonzero(np.isfinite(x) & np.isfinite(y))
        if ok.size < ZONE_MIN_VALID or np.unique(y[ok]).size < 2:
            continue
        ok = rng.choice(ok, size=min(ok.size, NUM_DATA_PTS), replace=False)
        models[z] = fit_idr(x[ok], y[ok])
        samples[z] = dict(x=x[ok], y=y[ok], day=ok // locs.size)
    return models, samples


NB = len(PROB_BINS) - 1


def verify(models, X, Y, clim=None, keep_demo=False):
    # Score raw, IDR and climatology on the test tables X, Y (days × grid boxes).
    # `clim` holds the IMERG climatology's CRPS (and event probability) for the same
    # cells; without it each zone's training-period climatology is used.
    out = {k: np.zeros(NPIX) for k in ("raw", "idr", "clim", "n")}
    daily = {k: np.zeros(X.shape[0]) for k in ("raw", "idr", "clim", "n")}
    rel = {k: dict(n=np.zeros(NB), obs=np.zeros(NB), prob=np.zeros(NB)) for k in ("raw", "idr", "clim")}
    bs = dict(raw=0.0, idr=0.0, clim=0.0)
    demo_sample = None
    for z, model in models.items():
        zp = np.flatnonzero(PIX_ZONE == z)
        x, y = X[:, zp], Y[:, zp]
        ok = np.isfinite(x) & np.isfinite(y)
        if clim is not None:
            ok &= np.isfinite(clim["crps"][:, zp])
        yi = np.where(ok, np.rint(y / RES), 0).astype(np.int64)
        tab = lookup_tables(model, yi.max())
        xi_raw = np.where(ok, np.rint(x / RES), 0).astype(np.int64)
        xi = np.clip(xi_raw, 0, tab["R"] - 1)

        raw = np.abs(x - y)                         # CRPS of a single-number forecast = absolute error
        idr = tab["crps"][xi, yi]
        ref = tab["crps_clim"][yi] if clim is None else clim["crps"][:, zp]
        for key, v in (("raw", raw), ("idr", idr), ("clim", ref)):
            out[key][zp] = np.where(ok, v, 0).sum(axis=0)
            daily[key] += np.where(ok, v, 0).sum(axis=1)
        out["n"][zp] = ok.sum(axis=0)
        daily["n"] += ok.sum(axis=1)

        # Heavy rain: event and raw yes/no on the same 0.1 mm/day levels as the model
        event = (yi > EV)[ok].astype(np.float64)
        p_raw = (xi_raw > EV)[ok].astype(np.float64)    # raw AIFS can only say 0 or 1
        p_idr = tab["p_event"][xi][ok].astype(np.float64)
        p_ref = (np.full(event.shape, tab["p_event_clim"]) if clim is None or clim["p_event"] is None
                 else clim["p_event"][:, zp][ok].astype(np.float64))
        for key, pk in (("raw", p_raw), ("idr", p_idr), ("clim", p_ref)):
            bs[key] += ((pk - event) ** 2).sum()
            b = np.clip(np.digitize(pk, PROB_BINS) - 1, 0, NB - 1)
            rel[key]["n"] += np.bincount(b, minlength=NB)
            rel[key]["obs"] += np.bincount(b, weights=event, minlength=NB)
            rel[key]["prob"] += np.bincount(b, weights=pk, minlength=NB)

        if keep_demo and z == DEMO_ZONE:            # test pairs from a random subset of the zone's boxes
            cols = np.sort(np.random.default_rng(SEED + 1).choice(zp.size, min(zp.size, DEMO_SAMPLE_BOXES),
                                                                   replace=False))
            m = ok[:, cols]
            dd, cc = np.nonzero(m)
            demo_sample = dict(x=x[:, cols][m], y=y[:, cols][m], clim=ref[:, cols][m] if clim is not None
                               else tab["crps_clim"][yi[:, cols][m]], day=dd)
    n = out.pop("n")
    maps = {k: np.where(n > 0, v / np.maximum(n, 1), np.nan) for k, v in out.items()}
    totals = {k: v.sum() / n.sum() for k, v in out.items()}
    totals.update({f"bs_{k}": v / n.sum() for k, v in bs.items()})
    zone_sums = {z: {k: out[k][PIX_ZONE == z].sum() / max(n[PIX_ZONE == z].sum(), 1)
                     for k in out} for z in models}
    nd = daily.pop("n")
    daily_sum = {k: v.copy() for k, v in daily.items()}
    daily = {k: np.where(nd > 0, v / np.maximum(nd, 1), np.nan) for k, v in daily.items()}
    return dict(maps=maps, totals=totals, zones=zone_sums, rel=rel, n=n, daily=daily,
                daily_sum=daily_sum, daily_n=nd, demo=demo_sample)

In [ ]:
rng = np.random.default_rng(SEED)
MODELS, SAMPLES, RESULTS = {}, {}, {}
for lead in LEAD_DAYS:
    t0 = time.time()
    X_tr, Y_tr, _, _ = paired(AIFS_TRAIN, OBS_TRAIN, OBS_TRAIN_DATES, lead)
    MODELS[lead], SAMPLES[lead] = fit_zone_models(X_tr, Y_tr, rng)
    del X_tr, Y_tr
    X_te, Y_te, dates, rows = paired(AIFS_TEST, OBS_TEST, OBS_TEST_DATES, lead)
    clim = None if CLIM_TEST is None else {k: (None if v is None else v[rows]) for k, v in CLIM_TEST.items()}
    RESULTS[lead] = verify(MODELS[lead], X_te, Y_te, clim, keep_demo=(lead == LEAD_DAYS[0]))
    RESULTS[lead]["dates"] = dates
    del X_te, Y_te
    t = RESULTS[lead]["totals"]
    print(f"lead day {lead}: {len(MODELS[lead])} zone models | {dates.size} test days "
          f"({dates[0]:%d %b} – {dates[-1]:%d %b %Y}) | mean CRPS raw {t['raw']:.3f}, "
          f"IDR {t['idr']:.3f}, climatology {t['clim']:.3f} mm/hr | {time.time() - t0:.0f} s")

## 6 · Skill, and the decomposition of the raw forecast's score

* **Left: mean CRPS** over all land grid boxes and test days (mm/hr; lower is better). For raw AIFS this is the mean absolute error.
* **Right: CRPS skill score** against IMERG climatology, $\mathrm{CRPSS} = 1 - \mathrm{CRPS}/\mathrm{CRPS}_{clim}$: 0% means no better than climatology, 100% means perfect.

A single-number forecast is heavily penalised by the CRPS whenever it is confidently wrong. That is why raw AIFS can score below climatology while the calibrated version, which states its uncertainty, beats it.

In [ ]:
summary = pd.DataFrame({
    lead: {"CRPS raw": r["totals"]["raw"], "CRPS IDR": r["totals"]["idr"],
           "CRPS climatology": r["totals"]["clim"],
           "CRPSS raw (%)": 100 * (1 - r["totals"]["raw"] / r["totals"]["clim"]),
           "CRPSS IDR (%)": 100 * (1 - r["totals"]["idr"] / r["totals"]["clim"]),
           "improvement IDR vs raw (%)": 100 * (1 - r["totals"]["idr"] / r["totals"]["raw"])}
    for lead, r in RESULTS.items()}).T
summary.index.name = "lead day"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
L = summary.index.values
for col, c, lab, ls in (("CRPS raw", ORANGE, "raw AIFS", "-"), ("CRPS IDR", BLUE, "calibrated (IDR)", "-"),
                        ("CRPS climatology", GREY, "climatology", "--")):
    ax1.plot(L, summary[col], color=c, lw=2, ls=ls, marker="o", ms=8, label=lab)
ax1.set(xlabel="lead day", ylabel="mean CRPS (mm/hr)", title="Error: lower is better", xticks=L)
ax1.set_ylim(bottom=0)
for col, c, lab in (("CRPSS raw (%)", ORANGE, "raw AIFS"), ("CRPSS IDR (%)", BLUE, "calibrated (IDR)")):
    ax2.plot(L, summary[col], color=c, lw=2, marker="o", ms=8, label=lab)
ax2.axhline(0, color=GREY, lw=1, ls="--")
ax2.set(xlabel="lead day", ylabel="CRPSS vs climatology (%)", title="Skill: higher is better", xticks=L)
ax1.legend(frameon=False, loc="lower left")
ax2.legend(frameon=False, loc="center left")
plt.tight_layout()
plt.show()
summary.round(3)

### MCB − DSC + UNC

The raw forecast's score splits into three parts:

$$\mathrm{CRPS}_{raw} = \underbrace{(\mathrm{CRPS}_{raw} - \mathrm{CRPS}_{cal})}_{\text{MCB: miscalibration}} - \underbrace{(\mathrm{CRPS}_{clim} - \mathrm{CRPS}_{cal})}_{\text{DSC: discrimination}} + \underbrace{\mathrm{CRPS}_{clim}}_{\text{UNC: uncertainty}}$$

* **MCB** is the error that calibration removes.
* **DSC** is the information beyond climatology. It is the term that matters for operational value: calibration can always remove MCB, and UNC is fixed by the climate.
* **UNC** is climatology's score: how hard the problem is.

It follows that $\mathrm{CRPSS}_{raw} = (\mathrm{DSC} - \mathrm{MCB})/\mathrm{UNC}$ and $\mathrm{CRPSS}_{cal} = \mathrm{DSC}/\mathrm{UNC}$.

**A caution for statisticians.** Here "cal" is the IDR forecast fitted on the *training* period, which is the honest choice for a forecast you could issue. The formal decomposition (Arnold et al.) refits on the evaluation data itself, which guarantees MCB ≥ 0 and DSC ≥ 0. With an out-of-sample recalibration, as here, either term can come out negative by chance.

In [ ]:
dec = pd.DataFrame({"MCB": summary["CRPS raw"] - summary["CRPS IDR"],
                    "DSC": summary["CRPS climatology"] - summary["CRPS IDR"],
                    "UNC": summary["CRPS climatology"]})
dec["MCB - DSC + UNC"] = dec["MCB"] - dec["DSC"] + dec["UNC"]
dec["CRPS raw"] = summary["CRPS raw"]
assert np.allclose(dec["MCB - DSC + UNC"], dec["CRPS raw"])     # the identity holds exactly

fig, ax = plt.subplots(figsize=(7, 3.6))
w = 0.25
ax.bar(L - w, dec["MCB"], w, color=ORANGE, label="MCB (removed by calibration)")
ax.bar(L, dec["DSC"], w, color=BLUE, label="DSC (information beyond climatology)")
ax.bar(L + w, dec["UNC"], w, color=GREY, label="UNC (climatology's CRPS)")
ax.set(xlabel="lead day", ylabel="mm/hr", xticks=L, title="Decomposition of the raw AIFS score")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()
dec.round(4)

### Skill through the test period

The same three forecasts day by day: CRPS averaged over all land grid boxes and smoothed with a 7-day running mean. Calibration often helps most in the rainy seasons, when there is most rain to get wrong. The gap between the climatology line and the calibrated line is the skill that AIFS adds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, lead in zip(axes, (LEAD_DAYS[0], LEAD_DAYS[-1])):
    r = RESULTS[lead]
    d = pd.DataFrame(r["daily"], index=r["dates"]).asfreq("D").rolling(7, center=True, min_periods=4).mean()
    for col, c, lab, ls in (("raw", ORANGE, "raw AIFS", "-"), ("idr", BLUE, "calibrated (IDR)", "-"),
                            ("clim", GREY, "climatology", "--")):
        ax.plot(d.index, d[col], color=c, lw=2, ls=ls, label=lab)
    ax.set(title=f"Lead day {lead}", ylabel="mean CRPS (mm/hr), 7-day mean")
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", rotation=30)
axes[0].legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

## 7 · Uncertainty with limited data

Every number so far comes from one training year and one test period. How much should we trust it? Three experiments on the demo zone at lead day 2:

* **7a · How much data does calibration need?** Refit IDR on a random stretch of *N* consecutive weeks of training data and score each fit on the same held-out pairs.
* **7b · How sure are we?** Bootstrap the skill score, resampling either single pairs (as if independent) or whole days (respecting that neighbouring grid boxes share the weather).
* **7c · Is the skill real?** Shuffle the forecasts in the training pairs before fitting. This breaks the forecast–observation link, so any remaining "skill" comes from the method, not from AIFS.

The held-out pairs are every 2026 day at a random subset of the zone's grid boxes.

In [ ]:
lead0 = LEAD_DAYS[0]
D = RESULTS[lead0]["demo"]
S = SAMPLES[lead0][DEMO_ZONE]
model0 = MODELS[lead0][DEMO_ZONE]
yi_te = np.rint(D["y"] / RES).astype(np.int64)
xi_te = np.rint(D["x"] / RES).astype(np.int64)


def mean_crps(model, xi, yi):
    tab = lookup_tables(model, yi.max())
    return tab["crps"][np.clip(xi, 0, tab["R"] - 1), yi]


crps_idr0 = mean_crps(model0, xi_te, yi_te)
ref0 = D["clim"]
print(f"Demo zone {KG_LEGEND[DEMO_ZONE][0]}, lead day {lead0}: {D['x'].size:,} test pairs on "
      f"{np.unique(D['day']).size} days; CRPSS of the full model {100 * (1 - crps_idr0.mean() / ref0.mean()):+.1f}%")

# --- 7a: learning curve in weeks of training data ---------------------------------------------------
n_train_days = int(S["day"].max()) + 1
max_weeks = int(np.ceil(n_train_days / 7))
weeks = sorted({w for w in (1, 2, 4, 8, 13, 26, max_weeks) if w <= max_weeks})
rng7 = np.random.default_rng(7)
curve = np.full((len(weeks), 5), np.nan)
for i, wk in enumerate(weeks):
    span = min(wk * 7, n_train_days)
    for r in range(5):
        start = int(rng7.integers(0, n_train_days - span + 1))
        m = (S["day"] >= start) & (S["day"] < start + span)
        if m.sum() < 20 or np.unique(np.rint(S["y"][m] / RES)).size < 2:
            continue
        curve[i, r] = 1 - mean_crps(fit_idr(S["x"][m], S["y"][m]), xi_te, yi_te).mean() / ref0.mean()

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.fill_between(weeks, 100 * np.nanmin(curve, 1), 100 * np.nanmax(curve, 1), color=BLUE, alpha=0.18, lw=0,
                label="range over 5 random stretches")
ax.plot(weeks, 100 * np.nanmedian(curve, 1), color=BLUE, lw=2, marker="o", label="median")
ax.axhline(0, color=GREY, ls="--", lw=1)
ax.set(xscale="log", xticks=weeks, xticklabels=[str(w) for w in weeks], xlabel="weeks of AIFS runs used for training",
       ylabel="CRPSS vs climatology (%)", title="7a · Skill with less training data")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# --- 7b: bootstrap by pairs vs by days ---------------------------------------------------------------
rngb = np.random.default_rng(11)
n_boot = 400
ud, inv = np.unique(D["day"], return_inverse=True)
s_day, r_day = np.bincount(inv, weights=crps_idr0), np.bincount(inv, weights=ref0)
boot_day = np.array([1 - (k * s_day).sum() / (k * r_day).sum()
                     for k in (np.bincount(rngb.integers(0, ud.size, ud.size), minlength=ud.size)
                               for _ in range(n_boot))])
boot_pair = np.empty(n_boot)
for b in range(n_boot):
    i = rngb.integers(0, crps_idr0.size, crps_idr0.size)
    boot_pair[b] = 1 - crps_idr0[i].sum() / ref0[i].sum()

# --- 7c: shuffle control -----------------------------------------------------------------------------
shuffled = np.array([1 - mean_crps(fit_idr(np.random.default_rng(100 + r).permutation(S["x"]), S["y"]),
                                   xi_te, yi_te).mean() / ref0.mean() for r in range(20)])
train_clim = 1 - lookup_tables(model0, yi_te.max())["crps_clim"][yi_te].mean() / ref0.mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(100 * boot_pair, bins=30, color=GREY, alpha=0.7, label="resample pairs (assumes independence)")
ax1.hist(100 * boot_day, bins=30, color=BLUE, alpha=0.6, label="resample whole days")
ax1.set(xlabel="CRPSS of the calibrated forecast (%)", ylabel="bootstrap samples", title="7b · How sure are we?")
ax1.legend(frameon=False, fontsize=9)
ax2.scatter(100 * shuffled, np.zeros_like(shuffled), color=GREY, label="AIFS shuffled before fitting", alpha=0.7)
ax2.scatter([100 * (1 - crps_idr0.mean() / ref0.mean())], [0.4], color=BLUE, s=80, label="real AIFS")
ax2.scatter([100 * train_clim], [-0.4], color="k", marker="D", label="training-period climatology")
ax2.axvline(0, color=GREY, ls="--", lw=1)
ax2.set(yticks=[], ylim=(-1, 1), xlabel="CRPSS vs IMERG climatology (%)", title="7c · Is the skill real?")
ax2.legend(frameon=False, fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()
q_day, q_pair = np.percentile(boot_day, [5, 95]), np.percentile(boot_pair, [5, 95])
print(f"90% interval resampling days:  {100 * q_day[0]:+.1f}% to {100 * q_day[1]:+.1f}%  ({ud.size} days)\n"
      f"90% interval resampling pairs: {100 * q_pair[0]:+.1f}% to {100 * q_pair[1]:+.1f}%  ({crps_idr0.size:,} pairs)\n"
      f"Shuffled forecasts: median CRPSS {100 * np.median(shuffled):+.1f}% "
      f"(training-period climatology {100 * train_clim:+.1f}%)")

**Reading the results.**

* **7a.** With only a few weeks of data the model has seen one season and few heavy-rain days, so skill is lower and varies with *which* weeks you happened to have. That spread is the uncertainty a forecaster faces when a new model arrives.
* **7b.** Thousands of pairs, but only a few hundred independent days: resampling pairs gives a falsely narrow interval. Resampling days is the honest version.
* **7c.** Shuffled forecasts score about as well as a climatology learned from one training year. **Calibration cannot create skill**; it only reveals skill the forecast already has.

## 8 · Maps: where does calibration help?

First, the three scores side by side for the first and last lead days, on one colour scale: where the calibrated map is lighter than the climatology map, the forecast beats climatology there. Then the **improvement** from calibration, `100 × (1 − CRPS_IDR / CRPS_raw)`, and the **skill** of each forecast against climatology. Blue means better.

In [ ]:
vmax = np.nanpercentile(RESULTS[LEAD_DAYS[0]]["maps"]["raw"], 98)
fields, titles = [], []
for lead in (LEAD_DAYS[0], LEAD_DAYS[-1]):
    m = RESULTS[lead]["maps"]
    fields += [to_map(m["raw"]), to_map(m["idr"]), to_map(m["clim"])]
    titles += [f"Day {lead}: raw AIFS (abs. error)", f"Day {lead}: calibrated CRPS", f"Day {lead}: climatology CRPS"]
map_panels(fields, titles, cmap="YlOrBr", vmin=0, vmax=vmax, label="mean CRPS (mm/hr)", ncols=3,
           suptitle="Mean CRPS over the 2026 test days (lighter is better)")

In [ ]:
imp = [to_map(100 * (1 - RESULTS[l]["maps"]["idr"] / RESULTS[l]["maps"]["raw"])) for l in LEAD_DAYS]
map_panels(imp, [f"Lead day {l}" for l in LEAD_DAYS], cmap="RdBu", vmin=-60, vmax=60,
           label="improvement over raw AIFS (%)", ncols=4, extend="both",
           suptitle="Error reduction from calibration (blue = calibrated is better)")

In [ ]:
# Skill against climatology: where does each forecast beat "the usual rain for this place and time of year"?
fields = []
for lead in (LEAD_DAYS[0], LEAD_DAYS[-1]):
    m = RESULTS[lead]["maps"]
    fields += [to_map(100 * (1 - m["raw"] / m["clim"])), to_map(100 * (1 - m["idr"] / m["clim"]))]
map_panels(fields, [f"Day {LEAD_DAYS[0]}: raw AIFS", f"Day {LEAD_DAYS[0]}: calibrated",
                    f"Day {LEAD_DAYS[-1]}: raw AIFS", f"Day {LEAD_DAYS[-1]}: calibrated"],
           cmap="RdBu", vmin=-60, vmax=60, label="CRPSS vs climatology (%)", ncols=4,
           extend="both", suptitle="Skill against climatology (blue = better than climatology)")

## 9 · Skill by climate zone

Calibration does not help every climate equally. Deserts (BWh) get very little rain, so errors are small to start with and skill is fragile. Wet tropical zones (Af, Am, Aw) usually gain the most. The first table shows the improvement over raw AIFS (%) for each zone and lead day. The second shows the skill of the calibrated forecast against climatology (CRPSS, %): near zero means AIFS adds little beyond the usual rain for that place and season.

In [ ]:
improve, skill = [], []
for z in ZONES:
    base = {"code": KG_LEGEND[z][0], "climate": KG_LEGEND[z][1], "grid boxes": int((PIX_ZONE == z).sum())}
    i_row, s_row = dict(base), dict(base)
    for lead in LEAD_DAYS:
        s = RESULTS[lead]["zones"].get(z)
        i_row[f"day {lead}"] = np.nan if s is None else 100 * (1 - s["idr"] / s["raw"])
        s_row[f"day {lead}"] = np.nan if s is None else 100 * (1 - s["idr"] / s["clim"])
    improve.append(i_row)
    skill.append(s_row)
lead_cols = [f"day {l}" for l in LEAD_DAYS]
as_table = lambda rows: pd.DataFrame(rows).set_index("code").sort_values("grid boxes", ascending=False)

display(as_table(improve).style.format("{:.0f}", subset=lead_cols)
        .background_gradient(cmap="Blues", subset=lead_cols, vmin=0, vmax=60)
        .set_caption("Error reduction from calibration, % (CRPS of IDR vs mean absolute error of raw AIFS)"))
display(as_table(skill).style.format("{:.0f}", subset=lead_cols)
        .background_gradient(cmap="RdBu", subset=lead_cols, vmin=-40, vmax=40)
        .set_caption("Calibrated AIFS vs IMERG climatology: CRPSS, % (blue = better than climatology)"))

## 10 · Heavy-rain probabilities: reliability, discrimination and value

From here on the forecast is a probability of a single event: **rain above `EVENT_MM_DAY` (10 mm/day)**. Three forecasts are compared:

* **calibrated AIFS (IDR):** the probability from the zone model;
* **climatology:** the share of the ~650 past days (±15 days of the date, 2000–2020) above the threshold;
* **raw AIFS:** only "no" (0%) or "yes" (100%).

### 10a · Reliability and the Brier skill score

Of all the times the forecast said 30%, did the event happen about 30% of the time? Points on the diagonal mean the probabilities can be taken at face value. The small bars show **how often** each probability was used (sharpness). A useful forecast is both reliable and willing to issue high or low probabilities when the weather supports it.

In [ ]:
STYLE = {"idr": (BLUE, "-", "calibrated (IDR)"), "clim": (GREY, "--", "climatology"),
         "raw": (ORANGE, "none", "raw AIFS (yes/no)")}
centres = (PROB_BINS[:-1] + PROB_BINS[1:]) / 2
fig = plt.figure(figsize=(16, 5))
grid = fig.add_gridspec(4, 3, hspace=0.15, wspace=0.28)
for col, lead in enumerate((LEAD_DAYS[0], LEAD_DAYS[-1])):
    ax = fig.add_subplot(grid[:3, col])
    use = fig.add_subplot(grid[3, col], sharex=ax)
    ax.plot([0, 1], [0, 1], color=GREY, lw=1, ls=":")
    for i, (key, (c, ls, lab)) in enumerate(STYLE.items()):
        rel = RESULTS[lead]["rel"][key]
        ok = rel["n"] >= 50                                     # skip sparsely used bins
        ax.plot(rel["prob"][ok] / rel["n"][ok], rel["obs"][ok] / rel["n"][ok], color=c, lw=2, ls=ls,
                marker="o", ms=7, label=lab)
        use.bar(centres - 0.015 + 0.015 * i, rel["n"] / rel["n"].sum(), width=0.015, color=c)
    ax.set(xlim=(0, 1), ylim=(0, 1), ylabel="observed frequency",
           title=f"Reliability, lead day {lead}: rain > {EVENT_MM_DAY:g} mm/day")
    ax.tick_params(labelbottom=False)
    use.set(yscale="log", ylim=(1e-4, 1), xlabel="forecast probability", ylabel="share of\nforecasts")
    if col == 0:
        ax.legend(frameon=False, loc="upper left")

ax = fig.add_subplot(grid[:, 2])
bss = pd.DataFrame({lead: {k: 100 * (1 - RESULTS[lead]["totals"][f"bs_{k}"]
                                     / RESULTS[lead]["totals"]["bs_clim"]) for k in ("raw", "idr")}
                    for lead in LEAD_DAYS}).T
for k, c, lab in (("raw", ORANGE, "raw AIFS (yes/no)"), ("idr", BLUE, "calibrated (IDR)")):
    ax.plot(bss.index, bss[k], color=c, lw=2, marker="o", ms=8, label=lab)
ax.axhline(0, color=GREY, lw=1, ls="--", label="climatology")
ax.set(xlabel="lead day", ylabel="Brier skill score vs climatology (%)", xticks=LEAD_DAYS,
       title=f"Skill for rain > {EVENT_MM_DAY:g} mm/day")
ax.legend(frameon=False, loc="center right", bbox_to_anchor=(1, 0.75))
plt.show()
bss.round(1).rename(columns={"raw": "BSS raw AIFS (%)", "idr": "BSS calibrated (%)"})

### 10b · Discrimination (ROC) and economic value (the cost–loss model)

**The decision problem.** A user can protect at cost $C$, or risk a loss $L$ if heavy rain arrives unprotected. Protecting costs $C$ for sure; not protecting costs $pL$ in expectation. So a user who trusts a probability $p$ should **protect when $p > C/L$**. Only the ratio $\alpha = C/L$ matters, with $0 < \alpha < 1$. Each user therefore turns a probability forecast into a yes/no decision at their own threshold, giving one 2×2 table per user.

**The value score** (Wilks, 2001) compares expected expenses per occasion, with base rate $\pi$:

$$VS(\alpha) = \frac{EE_{clim} - EE_{fc}}{EE_{clim} - EE_{perfect}},\qquad EE_{clim} = \min(\alpha, \pi),\quad EE_{perfect} = \alpha\pi,\quad EE_{fc} = \alpha\,(h + f) + m,$$

in units of $L$, where $h$, $f$ and $m$ are the shares of hits, false alarms and misses. $VS = 1$ is as valuable as a perfect forecast, $0$ no better than climatology, and below $0$ the user would do better ignoring the forecast. Users here take the probabilities at face value, so miscalibration is penalised.

**The ROC** plots the hit rate $h/\pi$ against the false-alarm rate $f/(1-\pi)$ for every warning threshold. It depends only on how the forecasts *rank* events against non-events, so it is unchanged by recalibration and shows **potential** rather than actual skill (Wilks, 2001). Both curves below are built from the same binned counts, with thresholds at the 0.05-wide bin edges.

In [ ]:
trapz = getattr(np, "trapezoid", None) or np.trapz   # numpy 2 renamed trapz


def tables_by_threshold(rel):
    # For each bin edge k, users act when the forecast is in bin k or above: shares of hits,
    # false alarms and misses (per occasion), from the binned counts.
    n, o = rel["n"], rel["obs"]
    N = n.sum()
    h = np.array([o[k:].sum() for k in range(NB + 1)]) / N
    f = np.array([(n - o)[k:].sum() for k in range(NB + 1)]) / N
    m = o.sum() / N - h
    return h, f, m, o.sum() / N


def value_score(h, f, m, pi, alpha):
    ee_fc = alpha * (h + f) + m
    ee_clim = np.minimum(alpha, pi)
    return (ee_clim - ee_fc) / (ee_clim - alpha * pi)


alphas = PROB_BINS[1:-1]               # users whose threshold sits at each interior bin edge
VS0 = {}                               # value curves at the first lead day, for 10c
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
auc = {}
for lead, ls in ((LEAD_DAYS[0], "-"), (LEAD_DAYS[-1], ":")):
    rels = RESULTS[lead]["rel"]
    for key, (c, _, lab) in STYLE.items():
        h, f, m, pi = tables_by_threshold(rels[key])
        hr, far = h / pi, f / (1 - pi)
        if key != "raw":
            order = np.argsort(far)
            auc[(lead, key)] = trapz(hr[order], far[order])
            axes[0].plot(far, hr, color=c, ls=ls, lw=2, label=f"{lab}, day {lead} (AUC {auc[(lead, key)]:.2f})")
            vs = value_score(h[1:-1], f[1:-1], m[1:-1], pi, alphas)
        else:
            k_yes = NB - 1                                     # raw "yes" falls in the top bin
            axes[0].plot(far[k_yes], hr[k_yes], "D", color=c, ms=8, label=f"{lab}, day {lead}" if ls == "-" else None)
            vs = value_score(h[k_yes], f[k_yes], m[k_yes], pi, alphas)   # one table for every user
        axes[1].plot(alphas, vs, color=c, ls=ls, lw=2, label=f"{lab}, day {lead}")
        if lead == LEAD_DAYS[0]:
            VS0[key] = vs
axes[0].plot([0, 1], [0, 1], color=GREY, lw=1, ls=":")
axes[0].set(xlabel="false-alarm rate", ylabel="hit rate", title="ROC: potential skill", xlim=(0, 1), ylim=(0, 1))
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
axes[1].axhline(0, color=GREY, lw=1)
axes[1].set(xscale="log", xlabel="cost/loss ratio C/L", ylabel="value score VS", ylim=(-0.5, 1),
            title="Value score: actual value for each user")
axes[1].legend(frameon=False, fontsize=8)

# 10c · collapsing the curve to one number: a beta-weighted average over users (lead day 2)
weights = {"uniform (a=b=1)": (1, 1), "cheap actions (a=1, b=7)": (1, 7),
           "mid-range (a=b=4)": (4, 4), "expensive actions (a=7, b=1)": (7, 1)}
rows = {}
for name, (a, b) in weights.items():
    wts = stats.beta(a, b).pdf(alphas)
    wts = wts / wts.sum()
    rows[name] = {k: float(np.nansum(wts * np.clip(VS0[k], -1, None))) for k in ("idr", "clim", "raw")}
wt = pd.DataFrame(rows).T.rename(columns={"idr": "calibrated", "clim": "climatology", "raw": "raw yes/no"})
for i, (name, (a, b)) in enumerate(weights.items()):
    axes[2].plot(alphas, stats.beta(a, b).pdf(alphas), lw=2, label=name)
axes[2].set(xlabel="cost/loss ratio C/L", ylabel="density", title="Assumed populations of users", yscale="log",
            ylim=(1e-3, 20))
axes[2].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()
print(f"Average value score at lead day {LEAD_DAYS[0]} under each population of users "
      "(VS floored at -1 so a few badly misled users don't dominate):")
wt.round(3)

**Three results worth stating** (Wilks, 2001):

* **Calibration protects users.** Perfectly calibrated probabilities have $VS \ge 0$ for every $C/L$. Negative value comes from miscalibration, here mostly the raw yes/no forecast, which imposes one threshold on every user.
* **The ROC shows potential skill only.** It would not change if the forecasts were recalibrated, while the value score would. This is the event-level counterpart of DSC versus MCB in Section 6.
* **One number means choosing a population of users.** Averaging $VS$ over a uniform distribution of $C/L$ is equivalent to the Brier score. Other weightings (beta densities above) can change which forecast looks best: compare the rows of the table.

## 11 · Discussion questions

1. What are the three components of the decomposition in Section 6, and which matters most for operational utility?
2. AIFS-Single gives one rainfall number per grid box and day, while climatology is a distribution. What terms describe these forecasts best, and are they comparable in skill?
3. How can we best measure AIFS-Single's discrimination against climatology?
4. AIFS-Single is new, with under a year of forecasts to recalibrate on. What can we do to make sure there are enough samples to train on? Use Section 7.
5. The 2026 test truth is IMERG Late, while training used IMERG Final. How would you check how much of the test error comes from that change of product?
6. Which population of users (Section 10c) should a national weather service optimise for, and who decides?

Short answers to questions 1–4 are in the companion note.

### References

* Henzi, A., Ziegel, J. F. and Gneiting, T. (2021). Isotonic distributional regression. *Journal of the Royal Statistical Society Series B*, 83(5), 963–993.
* Wilks, D. S. (2001). A skill score based on economic value for probability forecasts. *Meteorological Applications*, 8, 209–219.
* Richardson, D. S. (2000). Skill and economic value of the ECMWF ensemble prediction system. *Quarterly Journal of the Royal Meteorological Society*, 126, 649–667.
* Arnold, S., Walz, E.-M., Ziegel, J. and Gneiting, T. (2024). Decompositions of the mean continuous ranked probability score. *Electronic Journal of Statistics*.
* Beck, H. E. et al. (2023). High-resolution (1 km) Köppen–Geiger maps for 1901–2099 based on constrained CMIP6 projections. *Scientific Data*, 10, 724.

